# Pseudo-Zone Huber With Nearest-Analog Delta

Third alignment-track experiment. The earlier screens showed that nearest-analog target-zone transfer has useful signal, but fixed shrinkage is too blunt. This notebook treats the nearest-analog delta gap as a model feature and tests robust pseudo-zone residual heads.

## 1. Setup

In [1]:
"Configure imports, paths, and robust alignment-head candidates."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__pseudo_zone_huber"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

BASE_RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
ANALOG_SAMPLE_PATH = ARTIFACT_TABLES / "model__multi_analog_zone_blend_residual_sample__2026-06-23.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
ANALOG_SOURCE_CANDIDATE = "weighted_k1_s050"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 12.0

CANDIDATES = [
    {"name": "baseline_no_analog_head", "kind": "baseline", "alpha": 0.0, "shrink": 0.0},
    {"name": "fixed_nearest_s005", "kind": "fixed", "shrink": 0.05},
    {"name": "fixed_nearest_s010", "kind": "fixed", "shrink": 0.10},
    {"name": "fixed_nearest_s015", "kind": "fixed", "shrink": 0.15},
    {"name": "ridge_zone_a100_s050", "kind": "ridge", "alpha": 100.0, "shrink": 0.50},
    {"name": "ridge_zone_a300_s050", "kind": "ridge", "alpha": 300.0, "shrink": 0.50},
    {"name": "ridge_zone_a1000_s050", "kind": "ridge", "alpha": 1000.0, "shrink": 0.50},
    {"name": "ridge_poly2_a1000_s035", "kind": "ridge_poly2", "alpha": 1000.0, "shrink": 0.35},
    {"name": "huber_zone_a005_s050", "kind": "huber", "alpha": 0.05, "shrink": 0.50},
    {"name": "huber_zone_a010_s050", "kind": "huber", "alpha": 0.10, "shrink": 0.50},
    {"name": "huber_zone_a050_s050", "kind": "huber", "alpha": 0.50, "shrink": 0.50},
    {"name": "zone_local_ridge_a100_s075", "kind": "zone_local_ridge", "alpha": 100.0, "shrink": 0.75},
    {"name": "zone_local_huber_a010_s075", "kind": "zone_local_huber", "alpha": 0.10, "shrink": 0.75},
]

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"candidate_count={len(CANDIDATES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__pseudo_zone_huber
candidate_count=13


## 2. Load Baseline Residuals And Nearest-Analog Gap

In [2]:
"Load the current best residual sample and merge nearest-analog delta gap features."

base_all = pd.read_csv(BASE_RESIDUAL_PATH)
base = base_all.loc[base_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "distance_from_boundary_md", "eval_progress", "abs_projected_delta_10", "y_true", "y_pred", "residual"]:
    base[col] = pd.to_numeric(base[col], errors="coerce")
base["fold"] = base["fold"].astype(int)
base["row_idx"] = base["row_idx"].astype(int)
base["distance_log1p"] = np.log1p(base["distance_from_boundary_md"].clip(lower=0))
base["baseline_pred"] = base["y_pred"]
base["baseline_residual"] = base["y_true"] - base["baseline_pred"]
base["pseudo_zone"] = pd.cut(base["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])

analog = pd.read_csv(ANALOG_SAMPLE_PATH)
analog = analog.loc[analog["candidate"].eq(ANALOG_SOURCE_CANDIDATE), ["fold", "wellname", "row_idx", "analog_gap"]].copy()
for col in ["fold", "row_idx", "analog_gap"]:
    analog[col] = pd.to_numeric(analog[col], errors="coerce")
analog["fold"] = analog["fold"].astype(int)
analog["row_idx"] = analog["row_idx"].astype(int)

model_frame = base.merge(analog, on=["fold", "wellname", "row_idx"], how="left")
model_frame["analog_gap_clipped"] = model_frame["analog_gap"].clip(-120.0, 120.0)
model_frame["analog_gap_abs"] = model_frame["analog_gap_clipped"].abs()
model_frame["analog_gap_sign"] = np.sign(model_frame["analog_gap_clipped"].fillna(0.0))

print(f"rows={len(model_frame):,}")
print(f"wells={model_frame['wellname'].nunique():,}")
print(model_frame.groupby("pseudo_zone", observed=False).size().rename("rows_by_zone").to_string())
print(model_frame[["analog_gap", "baseline_residual"]].corr().to_string())

rows=5,400
wells=773
pseudo_zone
z0    1160
z1    1414
z2    1412
z3    1414
                   analog_gap  baseline_residual
analog_gap           1.000000           0.068002
baseline_residual    0.068002           1.000000


## 3. Feature Matrix

In [3]:
"Create pseudo-zone and interaction features for robust residual heads."

ROW_FEATURES = ["eval_progress", "distance_log1p", "abs_projected_delta_10"]
ANALOG_FEATURES = ["analog_gap_clipped", "analog_gap_abs", "analog_gap_sign"]
ZONE_COLUMNS = ["zone_z0", "zone_z1", "zone_z2", "zone_z3"]


def add_zone_features(frame):
    out = frame.copy()
    zones = pd.get_dummies(out["pseudo_zone"].astype(str), prefix="zone")
    for col in ZONE_COLUMNS:
        out[col] = zones[col] if col in zones.columns else 0.0
    for zone_col in ZONE_COLUMNS:
        out[f"analog_gap__x__{zone_col}"] = out["analog_gap_clipped"] * out[zone_col]
        out[f"progress__x__{zone_col}"] = out["eval_progress"] * out[zone_col]
    out["analog_gap__x__progress"] = out["analog_gap_clipped"] * out["eval_progress"]
    out["analog_gap__x__distance_log1p"] = out["analog_gap_clipped"] * out["distance_log1p"]
    out["analog_gap__x__slope_instability"] = out["analog_gap_clipped"] * out["abs_projected_delta_10"]
    return out

FEATURE_COLUMNS = ROW_FEATURES + ANALOG_FEATURES + ZONE_COLUMNS + [
    f"analog_gap__x__{zone_col}" for zone_col in ZONE_COLUMNS
] + [
    f"progress__x__{zone_col}" for zone_col in ZONE_COLUMNS
] + [
    "analog_gap__x__progress",
    "analog_gap__x__distance_log1p",
    "analog_gap__x__slope_instability",
]

model_features = add_zone_features(model_frame)
print(pd.DataFrame({"feature": FEATURE_COLUMNS}).to_string(index=False))

                         feature
                   eval_progress
                  distance_log1p
          abs_projected_delta_10
              analog_gap_clipped
                  analog_gap_abs
                 analog_gap_sign
                         zone_z0
                         zone_z1
                         zone_z2
                         zone_z3
          analog_gap__x__zone_z0
          analog_gap__x__zone_z1
          analog_gap__x__zone_z2
          analog_gap__x__zone_z3
            progress__x__zone_z0
            progress__x__zone_z1
            progress__x__zone_z2
            progress__x__zone_z3
         analog_gap__x__progress
   analog_gap__x__distance_log1p
analog_gap__x__slope_instability


## 4. Out-Of-Fold Evaluation

In [4]:
"Train pseudo-zone analog residual heads on other folds and score held-out folds."

start_time = time.time()
prediction_rows = []
metric_rows = []
for candidate in CANDIDATES:
    for fold in sorted(model_features["fold"].unique()):
        train_rows = model_features.loc[~model_features["fold"].eq(fold)].copy()
        valid_rows = model_features.loc[model_features["fold"].eq(fold)].copy()
        raw_correction = np.zeros(len(valid_rows), dtype=float)
        if candidate["kind"] == "fixed":
            raw_correction = valid_rows["analog_gap_clipped"].fillna(0.0).to_numpy()
        elif candidate["kind"] == "ridge":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] == "ridge_poly2":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("poly", PolynomialFeatures(degree=2, include_bias=False)), ("model", Ridge(alpha=candidate["alpha"]))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] == "huber":
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
            model.fit(train_rows[FEATURE_COLUMNS], train_rows["baseline_residual"])
            raw_correction = model.predict(valid_rows[FEATURE_COLUMNS])
        elif candidate["kind"] in ["zone_local_ridge", "zone_local_huber"]:
            raw_correction = np.zeros(len(valid_rows), dtype=float)
            for zone in ["z0", "z1", "z2", "z3"]:
                train_zone = train_rows.loc[train_rows["pseudo_zone"].astype(str).eq(zone)]
                valid_mask = valid_rows["pseudo_zone"].astype(str).eq(zone).to_numpy()
                if len(train_zone) < 100 or not valid_mask.any():
                    continue
                zone_features = ["analog_gap_clipped", "analog_gap_abs", "eval_progress", "distance_log1p", "abs_projected_delta_10", "analog_gap__x__progress"]
                if candidate["kind"] == "zone_local_ridge":
                    model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(alpha=candidate["alpha"]))])
                else:
                    model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler()), ("model", HuberRegressor(alpha=candidate["alpha"], epsilon=1.35, max_iter=500))])
                model.fit(train_zone[zone_features], train_zone["baseline_residual"])
                raw_correction[valid_mask] = model.predict(valid_rows.loc[valid_mask, zone_features])
        correction = np.clip(raw_correction * candidate.get("shrink", 1.0), -OFFSET_CLIP, OFFSET_CLIP)
        y_true = valid_rows["y_true"].to_numpy()
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        metric_rows.append(
            {
                "candidate": candidate["name"],
                "kind": candidate["kind"],
                "fold": int(fold),
                "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "pseudo_zone", "y_true", "baseline_pred", "baseline_residual", "eval_progress", "distance_log1p", "abs_projected_delta_10", "analog_gap_clipped"]].copy()
        out["candidate"] = candidate["name"]
        out["analog_head_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_head"] = out["y_true"] - out["y_pred"]
        prediction_rows.append(out)

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
summary = (
    fold_metrics.groupby(["candidate", "kind"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_analog_head"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.to_string(index=False))

                 candidate             kind  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
      ridge_zone_a100_s050            ridge  14.700422  0.428277 10.404800         0.129716        0.977650           12.000000    5400      773                      -0.038123
      ridge_zone_a300_s050            ridge  14.700813  0.428207 10.403027         0.126656        0.930053           12.000000    5400      773                      -0.037732
        fixed_nearest_s005            fixed  14.706869  0.406753 10.391269         0.016333        0.820072            4.233987    5400      773                      -0.031676
     ridge_zone_a1000_s050            ridge  14.707223  0.431202 10.402839         0.123364        0.813223           12.000000    5400      773                      -0.031322
      huber_zone_a050_s050            huber  14.714032  0.432111 10.422147         0.034667        0.948629           12

## 5. Diagnostics And Export

In [5]:
"Summarize the best pseudo-zone analog head and export artifacts."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_analog_head")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()

zone_summary = (
    best_predictions.groupby(["pseudo_zone"], observed=False)
    .agg(
        rmse_before=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        rmse_after=("residual_after_head", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        mean_correction=("analog_head_correction", "mean"),
        rows=("row_idx", "size"),
    )
    .reset_index()
)
zone_summary["rmse_delta"] = zone_summary["rmse_after"] - zone_summary["rmse_before"]

well_before = model_features.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_head", "mean"), mean_head_correction=("analog_head_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()

findings = pd.DataFrame(
    [
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}", "modeling_implication": "Promote only if pseudo-zone analog heads clear the 0.05 RMSE gate and fold diagnostics are stable."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "If positive versus fixed analog shrink, learned heads are overfitting or under-shrunk."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive means pseudo-zone heads add well-level noise."},
        {"finding": "best_zone_delta", "value": f"{zone_summary.sort_values('rmse_delta').iloc[0]['pseudo_zone']} delta {zone_summary.sort_values('rmse_delta').iloc[0]['rmse_delta']:.4f}", "modeling_implication": "Shows where analog transfer is most useful across evaluation progress."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_well_diagnostics__{DATE_STAMP}.csv"
zone_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_zone_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__pseudo_zone_huber_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__pseudo_zone_huber__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
zone_summary.to_csv(zone_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Pseudo-Zone Huber With Nearest-Analog Delta

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen tests whether nearest-analog deltas should be learned through pseudo-zone robust residual heads rather than fixed shrinkage. It is still a residual-sample diagnostic, not a Kaggle submission notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_pseudo_zone_huber",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(FEATURE_COLUMNS)),
            "notes": "Fold-safe nearest-analog delta as pseudo-zone Huber/Ridge residual-head feature",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, zone_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "analog_source_candidate": ANALOG_SOURCE_CANDIDATE, "offset_clip": OFFSET_CLIP, "candidates": CANDIDATES, "features": FEATURE_COLUMNS, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(zone_summary.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                        value                                                                               modeling_implication
       best_candidate ridge_zone_a100_s050 RMSE 14.7004 vs sample baseline 14.7385 Promote only if pseudo-zone analog heads clear the 0.05 RMSE gate and fold diagnostics are stable.
sample_baseline_delta                                 best candidate delta -0.0381             If positive versus fixed analog shrink, learned heads are overfitting or under-shrunk.
     well_bias_change                      median absolute well-bias delta -0.0374                           Negative is good; positive means pseudo-zone heads add well-level noise.
      best_zone_delta                                             z2 delta -0.0629                             Shows where analog transfer is most useful across evaluation progress.
pseudo_zone  rmse_before  rmse_after  mean_correction  rows  rmse_delta
         z0     7.

## 6. Conclusion

In [6]:
"Print final decision summary."

print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

best_candidate=ridge_zone_a100_s050
sample_baseline_rmse=14.7385
best_rmse=14.7004
delta_vs_sample_baseline=-0.0381
promotion_gate_cleared=False
